In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

client = genai.Client()

In [16]:
def dot_product(a, b):
    result = 0
    for i in range(len(a)):
        result += a[i] * b[i]
        # print(f"{a[i]}*{b[i]}={result}")
    return result

def euclidean_distance(a, b):
    sum_sq = 0
    for i in range(len(a)):
        sum_sq += (a[i] - b[i]) ** 2
        # print(f"{a[i]}*{b[i]}={sum_sq}")
    return sum_sq ** 0.5

def manhattan_distance(a, b):
    total = 0
    for i in range(len(a)):
        total += abs(a[i] - b[i])
    return total

def vector_norm(v):
    sum_sq = 0
    for x in v:
        sum_sq += x ** 2
        # print(f"{x}**2={sum_sq}")
    return sum_sq ** 0.5

def cosine_similarity(a, b):
    dot = dot_product(a, b)
    norm_a = vector_norm(a)
    norm_b = vector_norm(b)
    # print(norm_a, norm_b)
    return dot / (norm_a * norm_b)

In [5]:
with open('images/eiffel_day.jpeg', 'rb') as f:
    image_bytes = f.read()

result = client.models.embed_content(
    model='gemini-embedding-2-preview',
    contents=[
        types.Part.from_bytes(
            data=image_bytes,
            mime_type='image/png',
        ),
    ]
)

eiffel_day_emb = result.embeddings
eiffel_day_emb

[ContentEmbedding(
   values=[
     0.017688716,
     -0.012334593,
     -0.015927415,
     0.029680265,
     0.0322355,
     <... 3067 more items ...>,
   ]
 )]

In [6]:
with open('images/eiffel_night.jpeg', 'rb') as f:
    image_bytes = f.read()

result = client.models.embed_content(
    model='gemini-embedding-2-preview',
    contents=[
        types.Part.from_bytes(
            data=image_bytes,
            mime_type='image/png',
        ),
    ]
)

eiffel_night_emb = result.embeddings
eiffel_night_emb

[ContentEmbedding(
   values=[
     0.02851941,
     -0.024382437,
     -0.013791555,
     0.02845508,
     0.018740885,
     <... 3067 more items ...>,
   ]
 )]

In [14]:
a = eiffel_night_emb[0].values
b = eiffel_day_emb[0].values

In [17]:
print("Dot Product:")
print(dot_product(a, b))
print("Euclidean distance:")
print(euclidean_distance(a, b))
print("Manhattan distance:")
print(manhattan_distance(a, b))
print("Cosine similarity:")
print(cosine_similarity(a, b))

Dot Product:
0.9087713554991355
Euclidean distance:
0.4271502957749926
Manhattan distance:
18.896336225413783
Cosine similarity:
0.908771316340667


**Image Descriptions:**  
- *eiffel_day.jpeg*: A photo of the Eiffel Tower taken during the day.  
- *eiffel_night.jpeg*: A photo of the Eiffel Tower taken at night.

| Metric             | Value              | Similar or Not | Reliability Ranking (1=Most Reliable) |
|--------------------|-------------------|---------------|---------------------------------------|
| Dot Product        | 0.9088            | Similar       | 3                                     |
| Euclidean Distance | 0.4272            | Similar       | 2                                     |
| Manhattan Distance | 18.8963           | Similar       | 4                                     |
| Cosine Similarity  | 0.9088            | Similar       | 1                                     |

**Notes:**  
- Cosine similarity is generally the most reliable for comparing image embeddings, especially when embeddings are normalized.  
- Euclidean distance is also reliable, but less so if embeddings are not normalized.  
- Dot product and Manhattan distance are less commonly used for image similarity.

2. Aggregated embedding for text and image input

In [19]:
with open('images/dog.jpeg', 'rb') as f:
    image_bytes = f.read()

result = client.models.embed_content(
    model='gemini-embedding-2-preview',
    contents=[
        types.Content(
            parts=[
                types.Part(text="An image of a dog"),
                types.Part.from_bytes(
                    data=image_bytes,
                    mime_type='image/png',
                )
            ]
        )
    ]
)

# This produces one embedding
for embedding in result.embeddings:
    print(embedding.values)

[-0.011397961, 0.024591062, -0.0005099339, 0.0017956261, 0.042514667, -0.009880178, -0.006108072, -0.014696884, 0.023794724, -0.062438004, 0.0018424691, 0.024608083, -0.0054642553, 0.00087743247, 0.0053796894, -0.03608956, -0.0037446027, 0.013733959, 0.010684078, -0.0062920377, -0.0006154088, 0.0014455802, -0.013996572, -0.00231167, -0.013439809, -0.012746161, -0.012035567, -0.0019084873, -0.0037816602, 0.06352909, -0.024504574, -0.0008370648, -0.0068081343, 0.017453264, -0.008720495, 0.00061018823, 0.036301907, -0.011080303, -0.01331915, -0.008310334, 0.009122621, -0.013363379, -0.012402621, -0.0023709096, -0.03038615, 0.0014766861, -0.033052668, 0.017287735, 0.02023793, -0.006589892, 0.0022632682, 0.012787472, 0.013830684, -0.0007630458, 0.02852396, 0.012957393, -0.008876745, 0.021075841, -0.0062409844, -0.017846989, 0.003406907, -0.0084449565, 0.007848192, -0.021156013, -0.026313193, 0.0068999315, 0.006582375, -0.0048416974, -0.0116175655, -0.0004759661, -0.01260759, -0.013548695, -

In [ ]:
result.embeddings[0].values

3. Creates multiple embeddings in one embedding call

In [28]:
with open('images/dog.jpeg', 'rb') as f:
    image_bytes = f.read()

result = client.models.embed_content(
    model='gemini-embedding-2-preview',
    contents=[
        "The dog is cute",
        types.Part.from_bytes(
            data=image_bytes,
            mime_type='image/png',
        ),
    ]
)

# This produces two embeddings
for embedding in result.embeddings:
    print(embedding.values)

[-0.0021832879, 0.01565753, -0.008990876, 0.0044426634, 0.0048184907, -0.011400718, 0.01653249, 0.021557769, 0.017757224, -0.042947482, -0.00666046, 0.024392862, -0.004138883, -0.006142543, -0.015260766, -0.014303859, 0.0073504904, 0.012453639, -0.0011684508, 0.0019494604, -0.011929252, 0.0052103987, 0.0027621621, 0.00063558226, -0.00022153677, -0.0065518282, -0.014048589, -0.014813547, -0.015150913, 0.06673274, 0.00092735403, 0.0055173873, -0.022780364, -0.016558053, -0.00458726, -0.0064444477, 0.0007151765, -0.013631866, -0.019548932, -0.03129769, -0.010541575, -0.0064500105, -0.0031961065, 0.03540302, -0.0450052, -0.001335808, -0.026439816, -0.0049341614, -0.007102844, 0.0066666817, -0.011559849, 0.0012539682, 0.021329738, 0.01658024, 0.020200681, 0.013635065, 0.027262252, -0.014770296, -0.026116969, 0.0041101463, -0.0126212435, -0.034758992, 0.0064304345, -0.0041831, -0.021834401, 0.020049602, -0.0014721924, -0.0015811792, -0.0064993254, 0.04074363, -0.029372578, -0.014115071, 0.00

In [ ]:
result.embeddings[0].values
result.embeddings[1].values

2. Aggregated embedding for text and image input
3. Creates multiple embeddings in one embedding call
4. Embedding audio
5. Embedding video
6. Embedding documents